In [1]:
import datetime

# --- 辅助函数 ---
def reverse_complement(seq: str) -> str:
    comp = str.maketrans('ACGTNacgtn', 'TGCANtgcan')
    return seq.translate(comp)[::-1]

# 将 UMI 序列编码为十进制整数（A=1, T=2, C=3, G=4）
def encode_umi_decimal(umi):
    base_map = {'A': '1', 'T': '2', 'C': '3', 'G': '4','N':'5'}
    try:
        num_str = ''.join(base_map[b] for b in umi)
        return int(num_str)
    except KeyError:
        raise ValueError("UMI must contain only A/T/C/G.")
    
# 计算序列差异
def hamming(a: str, b: str) -> int:
    return sum(x != y for x, y in zip(a, b))

# 读取测序文件（按你的原始函数改造）
def extract_sequences_from_fastq(i, sample_name, fastq_file, primer1, primer2, output_fasta):
    """
    x_dict / y_dict: 4nt -> 索引 的字典
    tol: 每个4nt段允许的最大错配数(默认1,更稳健)
    """
    results = []
    with open(fastq_file, 'r') as fastq_in, open(output_fasta, 'w') as fasta_out:
        seq_id = 1   # 输出序列计数
        fp_id = 1    # 找到两条引物的计数
        stage_get = 1 # 找到正确坐标的计数
        seq_num = 1  # 总序列计数

        while True:
            header = fastq_in.readline().strip()
            if not header:
                break
            sequence = fastq_in.readline().strip()
            plus_line = fastq_in.readline().strip()
            quality = fastq_in.readline().strip()
            seq_num += 1

            if seq_num % 1000000 == 0:
                print(seq_num)
                print(datetime.datetime.now())

            # ---- 引物定位（正向）----
            primer1_pos = sequence.find(primer1)
            primer2_pos = sequence.find(primer2)
            s_fr = '1'

            # 如正向找不到两条引物，尝试反向互补；注意要同时翻转质量串
            if primer1_pos < 0 or primer2_pos < 0:
                sequence_rc = reverse_complement(sequence)
                quality_rc = quality[::-1]
                primer1_pos = sequence_rc.find(primer1)
                primer2_pos = sequence_rc.find(primer2)
                if primer1_pos >= 0 and primer2_pos >= 0:
                    sequence = sequence_rc
                    quality = quality_rc
                    s_fr = '2'
                else:
                    continue

            # 两条引物都找到才计数
            if primer1_pos > 0 and primer2_pos > 0 and primer2_pos-primer1_pos>50:
                fp_id += 1
            else:
                continue

            # ---- 坐标转换----
            # 边界检查：X在primer1前12nt；Y在primer2后12nt
            if primer1_pos > 12 and primer2_pos < len(sequence) - (len(primer2) + 12):
                umi_x_seq = sequence[primer1_pos-12:primer1_pos]  # 12nt
                umi_y_seq = sequence[primer2_pos+len(primer2): primer2_pos+len(primer2)+12]  # 12nt
                match_x1 = encode_umi_decimal(umi_x_seq[0:4])  
                match_x2 = encode_umi_decimal(umi_x_seq[4:8])  
                match_x3 = encode_umi_decimal(umi_x_seq[8:12])  
                match_y1 = encode_umi_decimal(umi_y_seq[0:4])  
                match_y2 = encode_umi_decimal(umi_y_seq[4:8]) 
                match_y3 = encode_umi_decimal(umi_y_seq[8:12]) 
                stage_get += 1
            else:
                continue

            # ---- 提取 UMI 与特征序列区域 ----
            extended_start = primer1_pos + len(primer1) + 6
            extended_end   = primer2_pos - 6
            umi_code1 = sequence[primer1_pos+len(primer1): primer1_pos+len(primer1)+6]
            umi_code2 = sequence[primer2_pos-6: primer2_pos]
            umi_1 = encode_umi_decimal(umi_code1)
            umi_2 = encode_umi_decimal(umi_code2)

            full_seq = sequence[extended_start:extended_end]
            # 质量切片：用与序列对齐的 quality（若已反向互补则已被翻转）
            # 这里你原来切的是 primer1 到 primer2+primer2_len 的质量区间
            full_qual = quality[extended_start:extended_end]

            high_quality_bases = sum(1 for q in full_qual if ord(q) - 33 >= 37)
            if len(full_qual) > 0 and (high_quality_bases / len(full_qual) >= 0.8):
                seq_id += 1
                results.append(
                    f">{s_fr}_{match_x1}_{match_x2}_{match_x3}_{match_y1}_{match_y2}_{match_y3}_{umi_1}_{umi_2}\n{full_seq}\n"
                )

        fasta_out.writelines(results)

    print(f"提取完成，结果保存为 {output_fasta}\n")
    print(f"样本{sample_name} 总序列数_{seq_num-1}")
    print(f"找到坐标数_{fp_id-1}")
    print(f"有坐标序列数_{stage_get-1}")
    print(f"输出序列数_{seq_id-1}\n")


In [2]:
# 引物接头
primer1 = 'AATGTGCCTCGCACGTATACGCGT'  # 例如 'AGCTAGC'
primer2 = 'GCTCGATGTAATCGCGGACGCCAC'  # 例如 'CGTACGT'

#sample_list = ['1a','1b','1s','2a','2b','2s','3a','3b','3s']
sample_list = ['1a']
raw_path = 'D:/CL/AFISH1/afish_data_20251009/flash_data/'
out_path = 'D:/CL/AFISH1/afish_data_20251009/part_data/'
for i in range(len(sample_list)): 
    sample_name = sample_list[i]   
    fastq_file = raw_path + sample_name + '_filtered.extendedFrags.fastq'  # 输入FASTQ文件路径
    output_fasta = out_path + sample_name + '_part_full_info.fasta'
    extract_sequences_from_fastq(i, sample_name, fastq_file, primer1, primer2, output_fasta)

1000000
2025-10-22 16:15:33.680303
2000000
2025-10-22 16:15:58.915639
3000000
2025-10-22 16:16:22.797810
4000000
2025-10-22 16:16:47.591987
5000000
2025-10-22 16:17:07.470905
6000000
2025-10-22 16:17:26.821093
7000000
2025-10-22 16:17:46.198206
8000000
2025-10-22 16:18:05.703185
9000000
2025-10-22 16:18:25.270361
10000000
2025-10-22 16:18:44.630131
11000000
2025-10-22 16:19:03.947641
12000000
2025-10-22 16:19:23.262572
13000000
2025-10-22 16:19:42.912114
14000000
2025-10-22 16:20:02.358881
15000000
2025-10-22 16:20:21.831276
16000000
2025-10-22 16:20:41.189106
17000000
2025-10-22 16:21:00.571278
18000000
2025-10-22 16:21:20.104483
19000000
2025-10-22 16:21:39.484389
20000000
2025-10-22 16:22:02.529917
21000000
2025-10-22 16:22:27.313970
22000000
2025-10-22 16:22:48.553549
23000000
2025-10-22 16:23:08.146134
24000000
2025-10-22 16:23:33.244175
25000000
2025-10-22 16:23:58.099374
26000000
2025-10-22 16:24:23.028271
27000000
2025-10-22 16:24:47.656427
28000000
2025-10-22 16:25:12.563412
2